In [1]:
# =============================================================================
# CATEGORIZACIÓN DE ESTACIONES BICING - VERSIÓN MEJORADA
# =============================================================================
# Usa el campo 'cross_street' (distrito/barrio) como fuente PRINCIPAL
# y keywords en el nombre como fuente SECUNDARIA para casos especiales.
#
# REGLA FUNDAMENTAL: Una estación solo tiene UNA categoría.
# =============================================================================

import pandas as pd
import json

# -----------------------------------------------------------------------------
# MAPEO DE BARRIOS A CATEGORÍAS
# -----------------------------------------------------------------------------
# Basado en el campo cross_street que tiene formato:
# "DD-Distrito/NN-Barrio"

BARRIO_A_CATEGORIA = {
    # =========================================================================
    # HOSPITAL - Zonas con hospitales principales
    # =========================================================================
    '07-Horta-Guinardó/41-la Vall d\'Hebron': 'hospital',  # Hospital Vall d'Hebron
    
    # =========================================================================
    # PLAYA - Zona litoral
    # =========================================================================
    '01-CiutatVella/03-la Barceloneta': 'playa',
    '10-SantMartí/67-la Vila Olímpica del Poblenou': 'playa',
    '01-CiutatVella/67-la Vila Olímpica del Poblenou': 'playa',
    '10-SantMartí/69-Diagonal Mar i el Front Marítim del Poblenou': 'playa',
    
    # =========================================================================
    # UNIVERSIDAD - Campus universitarios
    # =========================================================================
    '04-LesCorts/21-Pedralbes': 'universidad',  # Campus UB/UPC
    
    # =========================================================================
    # OFICINAS - Zona 22@ y distritos de negocios
    # =========================================================================
    '10-SantMartí/66-el Parc i la Llacuna del Poblenou': 'oficinas',  # 22@
    '10-SantMartí/68-el Poblenou': 'oficinas',                         # 22@
    '10-SantMartí/71-Provençals del Poblenou': 'oficinas',             # 22@
    '02-Eixample/66-el Parc i la Llacuna del Poblenou': 'oficinas',    # 22@
    
    # =========================================================================
    # COMERCIAL - Zonas de shopping y turismo
    # =========================================================================
    '01-CiutatVella/01-el Raval': 'comercial',
    '01-CiutatVella/02-el Barri Gòtic': 'comercial',
    '02-Eixample/02-el Barri Gòtic': 'comercial',
    '01-CiutatVella/04-Sant Pere, Santa Caterina i la Ribera': 'comercial',
    '02-Eixample/07-la Dreta de l\'Eixample': 'comercial',  # Passeig de Gràcia
    
    # =========================================================================
    # INDUSTRIAL - Zonas industriales y logísticas
    # =========================================================================
    '03-Sants-Montjuïc/12-la Marina del Prat Vermell': 'industrial',
    '03-Sants-Montjuïc/13-la Marina de Port': 'industrial',
    '09-SantAndreu/59-el Bon Pastor': 'industrial',
    '10-SantMartí/70-el Besòs i el Maresme': 'industrial',
    '10-SantMartí/73-la Verneda i la Pau': 'industrial',
    
    # =========================================================================
    # TRANSPORTE - Zonas de intercambiadores principales
    # =========================================================================
    '03-Sants-Montjuïc/15-Hostafrancs': 'transporte',  # Sants Estació
    '09-SantAndreu/61-la Sagrera': 'transporte',        # Sagrera AVE
    '10-SantMartí/65-el Clot': 'transporte',            # Estación Clot
    
    # =========================================================================
    # RESIDENCIAL - Todos los demás barrios
    # =========================================================================
    # Eixample
    '02-Eixample/05-el Fort Pienc': 'residencial',
    '02-Eixample/06-la Sagrada Família': 'residencial',
    '02-Eixample/08-l\'Antiga Esquerra de l\'Eixample': 'residencial',
    '02-Eixample/09-la Nova Esquerra de l\'Eixample': 'residencial',
    '02-Eixample/10-Sant Antoni': 'residencial',
    '02-Eixample/11-el Poble-sec': 'residencial',
    
    # Sants-Montjuïc
    '03-Sants-Montjuïc/11-el Poble-sec': 'residencial',
    '03-Sants-Montjuïc/14-la Font de la Guatlla': 'residencial',
    '03-Sants-Montjuïc/16-la Bordeta': 'residencial',
    '03-Sants-Montjuïc/17-Sants - Badal': 'residencial',
    '03-Sants-Montjuïc/18-Sants': 'residencial',
    
    # Les Corts
    '02-Eixample/19-les Corts': 'residencial',
    '04-LesCorts/19-les Corts': 'residencial',
    '04-LesCorts/20-la Maternitat i Sant Ramon': 'residencial',
    '04-LesCorts/23-Sarrià': 'residencial',
    
    # Sarrià-Sant Gervasi
    '05-Sarrià-StGervasi/23-Sarrià': 'residencial',
    '05-Sarrià-StGervasi/24-les Tres Torres': 'residencial',
    '05-Sarrià-StGervasi/25-Sant Gervasi - la Bonanova': 'residencial',
    '05-Sarrià-StGervasi/26-Sant Gervasi - Galvany': 'residencial',
    '05-Sarrià-StGervasi/27-el Putxet i el Farró': 'residencial',
    
    # Gràcia
    '06-Gràcia/28-Vallcarca i els Penitents': 'residencial',
    '06-Gràcia/30-la Salut': 'residencial',
    '06-Gràcia/31-la Vila de Gràcia': 'residencial',
    '06-Gràcia/32-el Camp d\'en Grassot i Gràcia Nova': 'residencial',
    
    # Horta-Guinardó
    '07-Horta-Guinardó/33-el Baix Guinardó': 'residencial',
    '07-Horta-Guinardó/35-el Guinardó': 'residencial',
    '07-Horta-Guinardó/36-la Font d\'en Fargues': 'residencial',
    '07-Horta-Guinardó/37-el Carmel': 'residencial',
    '07-Horta-Guinardó/39-Sant Genís dels Agudells': 'residencial',
    '07-Horta-Guinardó/42-la Clota': 'residencial',
    '07-Horta-Guinardó/43-Horta': 'residencial',
    
    # Nou Barris
    '08-NouBarris/44-Vilapicina i la Torre Llobeta': 'residencial',
    '08-NouBarris/45-Porta': 'residencial',
    '08-NouBarris/46-el Turó de la Peira': 'residencial',
    '08-NouBarris/48-la Guineueta': 'residencial',
    '08-NouBarris/49-Canyelles': 'residencial',
    '08-NouBarris/50-les Roquetes': 'residencial',
    '08-NouBarris/51-Verdun': 'residencial',
    '08-NouBarris/52-la Prosperitat': 'residencial',
    '08-NouBarris/53-la Trinitat Nova': 'residencial',
    '08-NouBarris/55-Ciutat Meridiana': 'residencial',
    
    # Sant Andreu
    '09-SantAndreu/57-la Trinitat Vella': 'residencial',
    '09-SantAndreu/58-Baró de Viver': 'residencial',
    '09-SantAndreu/60-Sant Andreu': 'residencial',
    '09-SantAndreu/62-el Congrés i els Indians': 'residencial',
    '09-SantAndreu/63-Navas': 'residencial',
    
    # Sant Martí
    '10-SantMartí/64-el Camp de l\'Arpa del Clot': 'residencial',
    '10-SantMartí/72-Sant Martí de Provençals': 'residencial',
}


# -----------------------------------------------------------------------------
# KEYWORDS PARA CASOS ESPECIALES (override del barrio)
# -----------------------------------------------------------------------------
# Algunas estaciones tienen ubicaciones especiales que no coinciden
# exactamente con su barrio. Estos keywords tienen PRIORIDAD sobre el barrio.

KEYWORDS_ESPECIALES = {
    'hospital': [
        'hospital',
        'clínic', 'clinic',
        'vall d\'hebron', 'vall hebron',
        'sant antoni maria claret',  # Hospital Sant Pau
    ],
    'transporte': [
        'arc triomf', 'arc de triomf',
        'estació sants', 'estació de sants',
        'pl. espanya', 'plaça espanya',
        'passeig de gràcia',  # Intercambiador principal
        'glòries', 'glories',
        'fabra i puig',
        'maria cristina',
    ],
    'playa': [
        'pg. marítim', 'pg. maritim', 'passeig marítim',
        'barceloneta',
        'av. litoral', 'av. del litoral',
        'joan de borbó',
        'port olímpic', 'port olimpic',
        'vila olímpica', 'vila olimpica',
        'garcia fària', 'garcia faria',
    ],
    'universidad': [
        'zona universitària', 'zona universitaria',
        'campus',
        'ramon trias fargas',  # UPF
        'facultat',
        'pl. universitat', 'plaça universitat',
    ],
}


def categorizar_estacion(row):
    """
    Asigna una categoría única a cada estación.
    
    Orden de prioridad:
    1. Keywords especiales en el nombre (para casos como hospitales)
    2. Mapeo de barrio a categoría
    3. Default: residencial
    """
    nombre = str(row.get('name', '') or row.get('nombre', '')).lower()
    barrio = str(row.get('cross_street', ''))
    
    # PASO 1: Verificar keywords especiales (máxima prioridad)
    for categoria, keywords in KEYWORDS_ESPECIALES.items():
        for kw in keywords:
            if kw.lower() in nombre:
                return categoria
    
    # PASO 2: Usar mapeo de barrio
    if barrio in BARRIO_A_CATEGORIA:
        return BARRIO_A_CATEGORIA[barrio]
    
    # PASO 3: Default
    return 'residencial'


def aplicar_categorizacion(df_estaciones):
    """
    Aplica la categorización a todas las estaciones.
    """
    df = df_estaciones.copy()
    
    # Normalizar nombres de columnas
    if 'nombre' not in df.columns and 'name' in df.columns:
        df['nombre'] = df['name']
    if 'id_estacion' not in df.columns and 'station_id' in df.columns:
        df['id_estacion'] = df['station_id']
    
    df['categoria'] = df.apply(categorizar_estacion, axis=1)
    return df


def resumen_categorizacion(df_categorizado):
    """
    Genera un resumen de la distribución de categorías.
    """
    resumen = df_categorizado['categoria'].value_counts().reset_index()
    resumen.columns = ['categoria', 'n_estaciones']
    resumen['porcentaje'] = (resumen['n_estaciones'] / resumen['n_estaciones'].sum() * 100).round(1)
    
    # Ordenar por número de estaciones
    orden = ['hospital', 'transporte', 'playa', 'universidad', 
             'oficinas', 'comercial', 'industrial', 'residencial']
    resumen['orden'] = resumen['categoria'].map({cat: i for i, cat in enumerate(orden)})
    resumen = resumen.sort_values('orden').drop('orden', axis=1)
    
    return resumen


def buscar_candidatos_por_zona(df_estaciones, n_ejemplos=5):
    """
    Muestra ejemplos de estaciones por cada categoría.
    Útil para validar la categorización.
    """
    df = aplicar_categorizacion(df_estaciones)
    
    print("\n" + "="*70)
    print("EJEMPLOS DE ESTACIONES POR CATEGORÍA")
    print("="*70)
    
    for categoria in ['hospital', 'transporte', 'playa', 'universidad', 
                      'oficinas', 'comercial', 'industrial', 'residencial']:
        estaciones = df[df['categoria'] == categoria]
        n_total = len(estaciones)
        
        print(f"\n{'─'*70}")
        print(f"📍 {categoria.upper()} ({n_total} estaciones)")
        print(f"{'─'*70}")
        
        ejemplos = estaciones[['id_estacion', 'nombre', 'cross_street']].head(n_ejemplos)
        if not ejemplos.empty:
            for _, row in ejemplos.iterrows():
                barrio_corto = row['cross_street'].split('/')[-1] if pd.notna(row['cross_street']) else 'N/A'
                print(f"  ID {row['id_estacion']:3d} | {row['nombre'][:45]:<45} | {barrio_corto}")


# =============================================================================
# DICCIONARIO SIMPLIFICADO PARA USO EN SCRIPTS EXISTENTES
# =============================================================================
# Compatible con tu código original del script de búsqueda

ZONAS_BUSCAR = {
    'hospital': [
        'hospital', 'clínic', 'clinic', 
        'vall d\'hebron', 'vall hebron',
        'sant antoni maria claret',
    ],
    'transporte': [
        'arc triomf', 'estació', 'pl. espanya',
        'passeig de gràcia', 'glòries', 'glories',
        'fabra i puig', 'maria cristina', 'clot',
    ],
    'playa': [
        'pg. marítim', 'pg. maritim', 'barceloneta',
        'av. litoral', 'joan de borbó', 'port olímpic',
        'vila olímpica', 'garcia fària',
    ],
    'universidad': [
        'zona universitària', 'campus', 'facultat',
        'ramon trias fargas', 'pl. universitat',
    ],
    'oficinas': [
        'ciutat de granada', 'pallars', 'taulat', 
        'llacuna', 'pere iv', 'sancho de ávila',
        'almogàvers', 'àlaba', 'pamplona',
    ],
    'comercial': [
        'pl. catalunya', 'rambla', 'portal',
        'via laietana', 'catedral', 'urquinaona',
    ],
    'industrial': [
        'zona franca', 'bon pastor', 'c/ 60',
        'besòs', 'verneda',
    ],
    'residencial': [
        'gràcia', 'lesseps', 'sarrià', 'sant gervasi',
        'horta', 'guinardó', 'nou barris', 'sant andreu',
    ],
}


# =============================================================================
# COLORES SUGERIDOS PARA DASHBOARD (coherentes con aurora_palette)
# =============================================================================
COLORES_CATEGORIAS = {
    'hospital':     '#FF006E',  # Rosa intenso (emergencia)
    'transporte':   '#3A86FF',  # Azul real (movimiento)
    'playa':        '#00F5D4',  # Turquesa (agua/mar)
    'universidad':  '#8338EC',  # Violeta (académico)
    'oficinas':     '#FF9100',  # Naranja (negocios)
    'comercial':    '#FEE440',  # Amarillo (shopping)
    'industrial':   '#6B7280',  # Gris (industrial)
    'residencial':  '#00BBF9',  # Azul brillante (hogar)
}


# =============================================================================
# EJECUCIÓN PRINCIPAL
# =============================================================================
if __name__ == "__main__":
    print("="*70)
    print("CATEGORIZACIÓN DE ESTACIONES BICING - VERSIÓN MEJORADA")
    print("="*70)
    
    try:
        with open('informacion_estaciones_bicing.json', 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        df = pd.DataFrame(data['data']['stations'])
        print(f"\n✅ Cargadas {len(df)} estaciones")
        
        # Aplicar categorización
        df_cat = aplicar_categorizacion(df)
        
        # Mostrar resumen
        print("\n" + "="*50)
        print("RESUMEN DE CATEGORIZACIÓN")
        print("="*50)
        resumen = resumen_categorizacion(df_cat)
        print(resumen.to_string(index=False))
        
        # Mostrar ejemplos
        buscar_candidatos_por_zona(df, n_ejemplos=5)
        
        # Verificar que no hay solapamientos
        print("\n" + "="*50)
        print("VERIFICACIÓN: ¿Hay estaciones duplicadas en categorías?")
        print("="*50)
        duplicados = df_cat.groupby('id_estacion')['categoria'].nunique()
        if (duplicados > 1).any():
            print("⚠️ HAY ESTACIONES CON MÚLTIPLES CATEGORÍAS")
        else:
            print("✅ Cada estación tiene exactamente UNA categoría")
            
    except FileNotFoundError:
        print("\n⚠️ Archivo JSON no encontrado.")
        print("\nDiccionario ZONAS_BUSCAR disponible:")
        for cat, keywords in ZONAS_BUSCAR.items():
            print(f"\n  '{cat}': {keywords}")


CATEGORIZACIÓN DE ESTACIONES BICING - VERSIÓN MEJORADA

⚠️ Archivo JSON no encontrado.

Diccionario ZONAS_BUSCAR disponible:

  'hospital': ['hospital', 'clínic', 'clinic', "vall d'hebron", 'vall hebron', 'sant antoni maria claret']

  'transporte': ['arc triomf', 'estació', 'pl. espanya', 'passeig de gràcia', 'glòries', 'glories', 'fabra i puig', 'maria cristina', 'clot']

  'playa': ['pg. marítim', 'pg. maritim', 'barceloneta', 'av. litoral', 'joan de borbó', 'port olímpic', 'vila olímpica', 'garcia fària']

  'universidad': ['zona universitària', 'campus', 'facultat', 'ramon trias fargas', 'pl. universitat']

  'oficinas': ['ciutat de granada', 'pallars', 'taulat', 'llacuna', 'pere iv', 'sancho de ávila', 'almogàvers', 'àlaba', 'pamplona']

  'comercial': ['pl. catalunya', 'rambla', 'portal', 'via laietana', 'catedral', 'urquinaona']

  'industrial': ['zona franca', 'bon pastor', 'c/ 60', 'besòs', 'verneda']

  'residencial': ['gràcia', 'lesseps', 'sarrià', 'sant gervasi', 'horta', 